# DQN

In this notebook, we will implement Deep Q-Learning Reinforcement learning algorithm for Lunar Lander Environment.

## Lunar Lander

This environment is a classic rocket trajectory optimization problem. The landing pad is always at coordinates (0,0). The state is an 8-dimensional vector: the coordinates of the lander in x & y, its linear velocities in x & y, its angle, its angular velocity, and two booleans that represent whether each leg is in contact with the ground or not.

There are four discrete actions available:<br>
- 0: do nothing<br>
- 1: fire left orientation engine<br>
- 2: fire main engine<br>
- 3: fire right orientation engine<br>

After every step a reward is granted. The total reward of an episode is the sum of the rewards for all the steps within that episode.

For each step, the reward:

- is increased/decreased the closer/further the lander is to the landing pad.

- is increased/decreased the slower/faster the lander is moving.

- is decreased the more the lander is tilted (angle not horizontal).

- is increased by 10 points for each leg that is in contact with the ground.

- is decreased by 0.03 points each frame a side engine is firing.

- is decreased by 0.3 points each frame the main engine is firing.

The episode receive an additional reward of -100 or +100 points for crashing or landing safely respectively.

An episode is considered a solution if it scores at least 200 points.


You can read more the LunarLander environment [here](https://gymnasium.farama.org/environments/box2d/lunar_lander/)

![LunarLander](https://gymnasium.farama.org/_images/lunar_lander.gif)

## Deep Q-Learning

The main idea behind Q-learning is that if we had a function $Q^*:\text{State}\times\text{Action}\to\mathbb{R}$ that could tell us what our return would be if we were to take an action in a given state, then we could easily construct a policy that maximizes our rewards:
$$
\pi^*(s)=\arg\max_a\,Q^*(s,a). \tag{1}
$$
But this is not scalable. We would need to compute $Q(s,a)$ for every state–action pair. If the state is, e.g., raw pixels of a game screen, it is computationally infeasible to cover the entire state space. Since neural networks are universal function approximators, we can create one and train it to resemble $Q^*$.

For our training update rule, every $Q$ function for a policy $\pi$ obeys the (policy) Bellman equation (expectations made explicit):
$$
Q^{\pi}(s,a)=\mathbb{E}_{\,s'\sim P(\cdot\mid s,a)}\!\Big[r(s,a)+\gamma\,\mathbb{E}_{\,a'\sim\pi(\cdot\mid s')}\big[Q^{\pi}(s',a')\big]\Big]. \tag{2}
$$

In DQN (off-policy), we use the optimal Bellman target (with a max over actions). The temporal-difference error $\delta$ is:
$$
\delta \;=\; Q(s,a;\theta)\;-\;\Big(r(s,a)\;+\;\gamma\,\max_{a'} Q_{\text{target}}(s',a')\Big). \tag{3}
$$

To minimise this error, we use the [Huber loss](https://en.wikipedia.org/wiki/Huber_loss), computed over a minibatch $B$ sampled from replay memory:
$$
\mathcal{L}(\theta)=\frac{1}{|B|}\sum_{(s,a,r,s')\in B}\,\mathcal{L}_{\text{Huber}}\!\big(\delta\big). \tag{4}
$$

$$
\mathcal{L}_{\text{Huber}}(\delta)=
\begin{cases}
\dfrac{1}{2}\,\delta^2, & \text{if } |\delta|\le 1,\\[4pt]
|\delta|-\dfrac{1}{2}, & \text{otherwise.}
\end{cases} \tag{5}
$$


In [ ]:
!pip install swig
!pip install "gymnasium[box2d]"

In [ ]:
import gymnasium as gym
import math
import random
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from collections import namedtuple
from itertools import count
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as T

In [ ]:
# if gpu is to be used
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Create the environment
env = gym.make("LunarLander-v3")

### Experience Replay

Learning from batches of consecutive samples is problematic as the sample are correlated and it can create a bad feedback loop if one action is dominated in the samples.

We can address these problems using an experience replay memory. It maintains a record for all the transitions experienced. The agent is then trained by sampling random minibatches from the replay memory.

### Q-Network

### Exploration vs Exploitation

Notice that Q-learning only learns about the states and actions it visits. What if an optimal state remains unvisited due to not being explored. The agent should sometimes pick suboptimal actions in order to visit new states and actions. <br>

A simple strategy is to use an $\epsilon$-greedy policy. According to this policy, the agent takes a random action with epsilon probability. The value of epsilon is high at the start of training and low towards the end. So, the agent explores more at the start and then exploit the learned policy more at the end.

### Hyperparameters

### Training

### Visualization

In [ ]:
# For visualization
from gym.wrappers.monitoring import video_recorder
from IPython.display import HTML
from IPython import display
import glob
import base64, io, os


os.environ['SDL_VIDEODRIVER'] = 'dummy'

In [ ]:


# Video folder
os.makedirs("video", exist_ok=True)

def show_video(env_name):
    """Display the saved MP4 inline."""
    mp4 = f'video/{env_name}.mp4'
    if os.path.exists(mp4):
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        display.display(HTML(data=f'''
            <video alt="evaluation" autoplay loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{encoded.decode('ascii')}" type="video/mp4" />
            </video>'''))
    else:
        print("Could not find video")

def show_video_of_model(env_name, greedy=True, seed=None):
    """
    Record one evaluation episode into video/<env_name>.mp4.
    - greedy=True: use the greedy action (no epsilon exploration) for smooth video.
      Set greedy=False to reuse your epsilon-greedy get_action().
    """
    # Create a fresh environment configured for frame capturing
    env_video = gym.make(env_name, render_mode="rgb_array")
    if seed is not None:
        # Optional: fix the seed for reproducibility
        obs, info = env_video.reset(seed=seed)
    else:
        obs, info = env_video.reset()

    # Prepare recorder
    vid = video_recorder.VideoRecorder(env_video, path=f"video/{env_name}.mp4")

    # Convert observation to float32 tensor with batch dimension
    state = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)

    for t in range(max_steps):
        # Capture current frame
        vid.capture_frame()

        # Choose action: greedy (no exploration) or your epsilon-greedy policy
        if greedy:
            with torch.no_grad():
                action = policy_net(state).max(1)[1].view(1, 1)  # tensor [[a]]
        else:
            action = get_action(state)  # uses global epsilon

        # Step the environment (Gymnasium API)
        obs, reward, terminated, truncated, info = env_video.step(action.item())
        done_bool = terminated or truncated

        # Next state tensor
        next_state = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)

        # Advance
        state = next_state

        if done_bool:
            break

    # Finalize
    vid.close()
    env_video.close()

# Record and display
show_video_of_model("LunarLander-v3", greedy=True, seed=42)
show_video("LunarLander-v3")


In [ ]:
import matplotlib.pyplot as plt

def plot_rewards(rewards, window=100):
    plt.figure(figsize=(12, 6))
    plt.plot(rewards, label='Total Reward per Episode')

    if len(rewards) >= window:
        moving_avg = np.convolve(rewards, np.ones(window)/window, mode='valid')
        plt.plot(range(window-1, len(rewards)), moving_avg, label=f'Moving Average (window={window})')

    plt.xlabel('Episode')
    plt.ylabel('Total Reward')
    plt.title('Total Reward vs Episode')
    plt.legend()
    plt.show()

plot_rewards(rewards)


In [ ]:
# Report (Gymnasium-compatible, greedy evaluation)
evaluation_episodes = 100
evaluation_rewards = []

for episode in range(evaluation_episodes):
    # Gymnasium reset -> (obs, info)
    obs, info = env.reset()
    # to float32 tensor + batch dim
    state = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)

    total_rewards = 0.0

    for _ in range(max_steps):
        # Greedy action from the policy network (no epsilon)
        with torch.no_grad():
            action = policy_net(state).max(1)[1].view(1, 1)  # tensor [[a]]

        # Gymnasium step -> (obs, reward, terminated, truncated, info)
        obs, reward, terminated, truncated, info = env.step(action.item())
        done = terminated or truncated

        total_rewards += reward

        # next state tensor
        state = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)

        if done:
            break

    evaluation_rewards.append(total_rewards)

mean_reward = float(np.mean(evaluation_rewards))
std_reward = float(np.std(evaluation_rewards))
print(f"Mean reward over {evaluation_episodes} episodes: {mean_reward:.2f}")
print(f"Std of rewards over {evaluation_episodes} episodes: {std_reward:.2f}")
